# Sufi Salma - Monte Carlo Particle Transport & Variance Reduction

**Implements analog and implicit capture (survival biasing) Monte Carlo simulations for particle transport through multi-layered shields. Optimizes the Figure of Merit (FOM) and minimizes relative uncertainty using Numba JIT compilation (`@njit`) for computational efficiency. Explores variance reduction techniques in deep-penetration and high-scattering regimes**

# Question 1

Part (a)

- Total cross section: $\Sigma_t = 1.0 \text{ cm}^{-1}$
- Shield thickness: $d = 5.0 \text{ cm}$
- Purely absorbing medium: $\Sigma_s = 0$, $\Sigma_a = \Sigma_t = 1.0 \text{ cm}^{-1}$

Derivation:

$$
P(x) = e^{-\Sigma_t x}
$$


Transmission Probability:


$$
P_{\text{trans}} = e^{-\Sigma_t \cdot d} = e^{-1.0 \cdot 5.0} = e^{-5}
$$


$$
P_{\text{trans}} = e^{-5} \approx 0.006737947
$$

Reflection Probability:

no scattering and the source is normal to the surface

$$
P_{\text{refl}} = 0
$$

Absorption Probability:

$$
P_{\text{abs}} = 1 - P_{\text{trans}} = 1 - e^{-5}
$$

$$
P_{\text{abs}} \approx 1 - 0.006737947 = 0.993262053
$$

Summary:
- Transmission: 0. ---> only about 0.67% while the mean free path is 1.0 cm, so it shows that particle interact in the first few centimeters.
- Reflection: ~ 0 ---> No particle bounce back to the source side because there is no scattering
- Absorption: 0.993262053 ---> the majority of particles are absorbed because in here we use purely absorbing shield and the thickness is quite enough to absorb the particles.

In [ ]:
#part b
import numpy as np

#sampling methods
def sample_distance_mfp(sigma_t):
    xi = np.random.uniform(1e-10, 1.0)
    mfp = -np.log(xi)
    return mfp / sigma_t

def sample_distance_path(sigma_t):
    xi = np.random.uniform(1e-10, 1.0)
    return -np.log(xi) / sigma_t

def run_transport(n_particles, thicknesses, sigma_ts, scatter_probs, method='mfp'):
    n_trans, n_ref, n_abs = 0, 0, 0
    boundaries = [0]
    for t in thicknesses: boundaries.append(boundaries[-1] + t)
    total_thick = boundaries[-1]

    for _ in range(n_particles):
        x, mu, alive = 0.0, 1.0, True

        while alive:
            region = next((i for i, t in enumerate(thicknesses)
                          if boundaries[i] <= x < boundaries[i+1]), None)

            if region is None:
                if x >= total_thick: n_trans += 1
                elif x < 0: n_ref += 1
                alive = False
                continue

            # properties
            sig_t = sigma_ts[region]
            p_scat = scatter_probs[region]

            # Sample distance
            dist = sample_distance_mfp(sig_t) if method == 'mfp' else sample_distance_path(sig_t)
            x_new = x + dist * mu

            # Check Boundaries
            if mu > 0 and x_new >= boundaries[region+1]:
                x = boundaries[region+1]
                if x >= total_thick:
                    n_trans += 1; alive = False
                continue

            # Left Boundary
            elif mu < 0 and x_new <= boundaries[region]:
                x = boundaries[region]
                if x <= 0:
                    n_ref += 1; alive = False
                continue

            # Collision
            x = x_new
            if np.random.uniform() < p_scat:
                mu = np.random.uniform(-1, 1)
            else:
                n_abs += 1; alive = False

    return n_trans, n_ref, n_abs

In [ ]:
#part b
N = 100000
thick = [5.0]
sigma = [1.0]
scat = [0.0]

t_mfp, r_mfp, a_mfp = run_transport(N, thick, sigma, scat, method='mfp')
t_path, r_path, a_path = run_transport(N, thick, sigma, scat, method='path')

# Calculate Probabilities
p_t_mc = t_mfp / N
p_r_mc = r_mfp / N
p_a_mc = a_mfp / N

p_t_analytical = 0.006737947
p_a_analytical = 1 - p_t_analytical

error = abs(p_t_mc - p_t_analytical) / p_t_analytical

print("Part b results:")
print(f"Transmission (analytical): {p_t_analytical:.6f}")
print(f"Transmission (mean free path): {p_t_mc:.6f}")
print(f"Transmission (distance path): {t_path/N:.6f}")

print(f"\nAbsorption (analytical): {p_a_analytical:.6f}")
print(f"Absorption (mean free path): {p_a_mc:.6f}")
print(f"Absorption (distance path): {a_path/N:.6f}")

print(f"\nReflection (mean free path): {p_r_mc:.6f}")
print(f"Reflection (distance path): {r_path/N:.6f}")

print(f"\nRelative Error (Transmission): {error:.4%}")

Analysis

- Mean free path length and free path length give similar results for $P_T$, $P_R$, and $P_A$, that means we can use both are correct and we can use for this case.

- The analytical value $P_T^{\text{analytical}} = e^{-5} \approx 0.006737947$ which is
about $\pm 1\sigma$ of MC estimate. It confirms that the code can model pure absorption accurately.

- For reflection, $P_R^{\text{MC}} $ ~ 0 validates that the MC implement the boundaries correctly, because in a purely absorbing medium with normal incidence ($\mu = +1$), no physical mechanism exists to reverse a particle's direction.

In [ ]:
#part c
#Scattering Probability = 10%
scat_c = [0.1]

t_c, r_c, a_c = run_transport(N, thick, sigma, scat_c, method='mfp')

print(f"Part c results (10% Scatter)")
print(f"Transmission: {t_c/N:.6f}")
print(f"Reflection:   {r_c/N:.6f}")
print(f"Absorption:   {a_c/N:.6f}")
print(f"Sum Check:    {(t_c + r_c + a_c)/N:.6f}")

Analysis

- Reflection increases, ($P_R > 0$). In 10% isotropic scattering case, particles can now reverse direction when $\mu$ is sampled from $\mathcal{U}[-1, 1]$. It allows particles to exit through the source side ($x < 0$), it makes $P_R^{\text{MC}} > 0$. This behavior is physically consistent and impossible in the purely absorbing case.

- Transmission increases, $P_T^{\text{MC}}$ is higher than in Part (b), because scattering allows more particles to survive collisions and travel through the shield.

- Absorption decreases, $P_A^{\text{MC}}$ decreases as there is more collisions (scattering),and either transmit or reflect.

- The tallies satisfy:
  $$
  P_T + P_R + P_A = \frac{N_{\text{trans}} + N_{\text{ref}} + N_{\text{abs}}}{N} \approx 1
  $$
  It confirms that there are no particles lost or duplicated in the simulation.

# Question 2

In [ ]:
!pip install numba

In [ ]:
import numba
import time
import numpy as np

@njit
def sample_distance_mfp(sigma_t):
    """Mean Free Path Approach: Sample λ, then s = λ/Σt"""
    xi = np.random.uniform(1e-10, 1.0)
    mfp = -np.log(xi)
    return mfp / sigma_t

@njit
def sample_distance_path(sigma_t):
    """Path Length Approach: Sample s directly"""
    xi = np.random.uniform(1e-10, 1.0)
    return -np.log(xi) / sigma_t

@njit
def run_transport(n_particles, boundaries, sigma_ts, scatter_probs, method):
    """
    method = 0: Mean Free Path
    method = 1: Path Length
    """
    n_trans, n_ref, n_abs = 0, 0, 0
    n_regions = len(boundaries) - 1
    tol = 1e-10
    max_steps = 10000

    for _ in range(n_particles):
        x = 0.0
        mu = 1.0
        alive = True
        steps = 0

        while alive and steps < max_steps:
            steps += 1

            region = -1
            for r in range(n_regions):
                if boundaries[r] - tol <= x < boundaries[r+1] + tol:
                    region = r
                    break

            if region == -1:
                if x >= boundaries[-1] - tol:
                    n_trans += 1
                elif x <= boundaries[0] + tol:
                    n_ref += 1
                else:
                    n_abs += 1
                alive = False
                continue

            sig_t = sigma_ts[region]
            p_scat = scatter_probs[region]


            if method == 0:
                dist = sample_distance_mfp(sig_t)
            else:
                dist = sample_distance_path(sig_t)

            x_new = x + dist * mu


            if mu > 0 and x_new >= boundaries[-1] - tol:
                n_trans += 1
                alive = False
                continue
            elif mu < 0 and x_new <= boundaries[0] + tol:
                n_ref += 1
                alive = False
                continue

            # Boundaries
            crossed = False
            if mu > 0 and x_new >= boundaries[region+1] - tol:
                x = boundaries[region+1]
                crossed = True
            elif mu < 0 and x_new <= boundaries[region] + tol:
                x = boundaries[region]
                crossed = True

            if crossed:
                continue  # Re-evaluate region next iteration

            # Collision
            x = x_new
            if np.random.uniform() < p_scat:
                mu = np.random.uniform(-1, 1)
            else:
                n_abs += 1
                alive = False

    return n_trans, n_ref, n_abs


def calc_uncertainty(p, n):
    if p == 0 or p == 1 or n == 0:
        return 0.0
    return np.sqrt(p * (1 - p) / n)

def run_case(name, thicknesses, sigma_ts, scatter_probs, n_list):
    print(f"\n{'-'*60}")
    print(f"{name}")
    print(f"{'-'*60}")
    print(f"Regions: {len(thicknesses)}")
    print(f"Thicknesses: {thicknesses} cm")
    print(f"Sigma_t: {sigma_ts} 1/cm")
    print(f"Scatter Prob: {scatter_probs}")
    print(f"{'-'*60}")

    # boundaries
    boundaries = [0.0]
    for t in thicknesses:
        boundaries.append(boundaries[-1] + t)

    results = []

    for n in n_list:
        print(f"\n--- N = {n:,} histories ---")
        start_time = time.time()

        t, r, a = run_transport_fast(n, np.array(boundaries),
                                      np.array(sigma_ts),
                                      np.array(scatter_probs))

        elapsed = time.time() - start_time

        p_t, p_r, p_a = t/n, r/n, a/n
        s_t, s_r, s_a = calc_uncertainty(p_t, n), calc_uncertainty(p_r, n), calc_uncertainty(p_a, n)

        print(f"  Transmission = {p_t:.6f} ± {s_t:.6f}")
        print(f"  Reflection = {p_r:.6f} ± {s_r:.6f}")
        print(f"  Absorption = {p_a:.6f} ± {s_a:.6f}")
        print(f"  Sum = {p_t + p_r + p_a:.6f}")

In [ ]:
def calc_uncertainty(p, n):
    if p == 0 or p == 1 or n == 0:
        return 0.0
    return np.sqrt(p * (1 - p) / n)

def run_full_case(name, thick, sigma, scat, n_list):
    """Runs BOTH sampling methods for each N"""
    boundaries = [0.0]
    for t in thick:
        boundaries.append(boundaries[-1] + t)

    print(f"\n{'='*80}")
    print(f"{name}")
    print(f"{'='*80}")
    print(f"Scatter Prob: {scat}")
    print(f"{'N':>12} {'Method':>6} {'P_T':>10} {'σ_T':>10} {'P_R':>10} {'σ_R':>10} {'P_A':>10} {'σ_A':>10} {'Sum':>8}")
    print("-"*80)

    results = []
    for n in n_list:
        for method_name, method_id in [('MFP', 0), ('Path', 1)]:
            t, r, a = run_transport(n, np.array(boundaries),
                                     np.array(sigma), np.array(scat), method_id)

            pt, pr, pa = t/n, r/n, a/n
            st, sr, sa = calc_uncertainty(pt, n), calc_uncertainty(pr, n), calc_uncertainty(pa, n)

            print(f"{n:12,} {method_name:6} {pt:10.6f} {st:10.6f} {pr:10.6f} {sr:10.6f} {pa:10.6f} {sa:10.6f} {pt+pr+pa:8.4f}")

            results.append({'N': n, 'Method': method_name, 'PT': pt, 'ST': st,
                           'PR': pr, 'SR': sr, 'PA': pa, 'SA': sa, 'Sum': pt+pr+pa})
    return results

In [ ]:
# Case 1: 0.1 → 10 → 100
c1_thick = [0.1, 0.1, 0.1]
c1_sigma = [0.1, 10.0, 100.0]
c1_scat  = [(0.1-0.01)/0.1, (10-0.1)/10, (100-10)/100]  # [0.1, 0.99, 0.9]

# Case 2: 100 → 10 → 0.1 (Swap 1 & 3)
c2_thick = [0.1, 0.1, 0.1]
c2_sigma = [100.0, 10.0, 0.1]
c2_scat  = [(100-10)/100, (10-0.1)/10, (0.1-0.01)/0.1]  # [0.9, 0.99, 0.1]

# Case 3: Remove Region 3
c3_thick = [0.1, 0.1]
c3_sigma = [0.1, 10.0]
c3_scat  = [(0.1-0.01)/0.1, (10-0.1)/10]  # [0.1, 0.99]

n_values = [1_000, 100_000]  # Histories 10_000_000 is separated

r1 = run_full_case("CASE 1: Σ_t = 0.1 → 10 → 100 1/cm", c1_thick, c1_sigma, c1_scat, n_values)
r2 = run_full_case("CASE 2: Σ_t = 100 → 10 → 0.1 1/cm", c2_thick, c2_sigma, c2_scat, n_values)
r3 = run_full_case("CASE 3: Σ_t = 0.1 → 10 1/cm (No R3)", c3_thick, c3_sigma, c3_scat, n_values)

Now, we run the same code for 10,000,000 histories (but it will take awhile to run, that is why I separate the cells)

In [ ]:
n_values = [10_000_000]  # Histories 10_000_000 is separated

r1 = run_full_case("CASE 1: Σ_t = 0.1 → 10 → 100 1/cm", c1_thick, c1_sigma, c1_scat, n_values)
r2 = run_full_case("CASE 2: Σ_t = 100 → 10 → 0.1 1/cm", c2_thick, c2_sigma, c2_scat, n_values)
r3 = run_full_case("CASE 3: Σ_t = 0.1 → 10 1/cm (No R3)", c3_thick, c3_sigma, c3_scat, n_values)

Analysis

- In case 1, the density increases ($\Sigma_t: 0.1 \to 10 \to 100$).

  As N increases, the statistical uncertainties decreases by 10. As the particles penetrate to a highly absorbing layer, the transmissions becomes low and reflection is not zero because there is high scattering probability in Region 2. The particles build up population before as they pass through the lightest shield before it passes the hardest one in region 3.

- In case 2, the density decreases, and $P_T$ is way lower than in Case 1. The ordering effect can be seen as  the particles pass the hardest shield first, most of the particles are either absorbed or scattered immediately.

- In case 3, when we remove Region 3, there is an impact to transmission that increases significantly because the final barrier which has high-probability of absoption is no longer exist.

- The path length and mean free path still gives similar results and for rare events, the relative uncertainty is still large even at N = 10,000,000. In all cases, $P_T + P_R + P_A \approx 1.0$ within statistical noise, it validates the particle conservation in the code.

# Question 3

In [ ]:
import numpy as np
!pip install numba
from numba import njit
import time

@njit
def run_transport_angle(n_particles, thickness, sigma_t, scatter_prob, mu_source):
    """
    Analog Monte Carlo with angled source.
    Returns counts (not probabilities) for FOM calculation.
    """
    n_trans, n_ref, n_abs = 0, 0, 0
    max_steps = 10000

    for _ in range(n_particles):
        x = 0.0
        mu = mu_source
        alive = True
        steps = 0

        while alive and steps < max_steps:
            steps += 1

            # Exits
            if mu > 0 and x >= thickness:
                n_trans += 1; alive = False; continue
            elif mu < 0 and x <= 0:
                n_ref += 1; alive = False; continue

            # Sample distance to collision
            xi = np.random.uniform(1e-10, 1.0)
            dist = -np.log(xi) / sigma_t
            x_new = x + dist * mu

            # Boundary crossing
            if mu > 0 and x_new >= thickness:
                n_trans += 1; alive = False; continue
            elif mu < 0 and x_new <= 0:
                n_ref += 1; alive = False; continue

            # Collision inside shield
            x = x_new
            if np.random.uniform() < scatter_prob:
                mu = np.random.uniform(-1, 1)
            else:
                n_abs += 1; alive = False

    return n_trans, n_ref, n_abs

In [ ]:
#Relative Uncertainty & FOM
def calc_relative_uncertainty(p, n):
    """R = σ/P for binomial process"""
    if p == 0 or p == 1 or n == 0:
        return 1.0
    sigma = np.sqrt(p * (1 - p) / n)
    return sigma / p

def calc_fom(r_max, time_sec):
    """Figure of Merit: FOM = 1 / (R_max² · T)"""
    if r_max == 0 or time_sec == 0:
        return 0.0
    return 1.0 / (r_max**2 * time_sec)

def run_adaptive(thickness, sigma_t, scatter_prob, mu_source,
                 max_histories=10_000_000, target_R=0.10, batch=50_000):
    total_trans, total_ref, total_abs = 0, 0, 0
    n_done = 0
    start_time = time.time()

    while n_done < max_histories:
        t, r, a = run_transport_angle(batch, thickness, sigma_t, scatter_prob, mu_source)
        total_trans += t
        total_ref += r
        total_abs += a
        n_done += batch

        # Probabilities and uncertainties
        p_t = total_trans / n_done
        p_r = total_ref / n_done
        p_a = total_abs / n_done

        r_t = calc_relative_uncertainty(p_t, n_done)
        r_r = calc_relative_uncertainty(p_r, n_done)
        r_a = calc_relative_uncertainty(p_a, n_done)
        r_max = max(r_t, r_r, r_a)

        if r_max <= target_R:
            break

    # Final calculations
    elapsed = time.time() - start_time
    p_t = total_trans / n_done
    p_r = total_ref / n_done
    p_a = total_abs / n_done

    r_t = calc_relative_uncertainty(p_t, n_done)
    r_r = calc_relative_uncertainty(p_r, n_done)
    r_a = calc_relative_uncertainty(p_a, n_done)
    r_max = max(r_t, r_r, r_a)

    fom = calc_fom(r_max, elapsed)

    return {
        'N': n_done, 'P_T': p_t, 'P_R': p_r, 'P_A': p_a,
        'R_T': r_t, 'R_R': r_r, 'R_A': r_a, 'R_max': r_max,
        'FOM': fom, 'Time': elapsed}

In [ ]:
# Parameters: Σt·d = 10 → if d=1cm, then Σt=10 cm⁻¹
thickness = 1.0  # cm
sigma_t = 10.0   # cm⁻¹ (so Σt·d = 10)

# Angles to test: θ = 0°, 30°, 60° → μ = cos(θ)
angles_deg = [0.0, 30.0, 60.0]
mus = [np.cos(np.radians(a)) for a in angles_deg]

# Two scattering ratios
cases = [
    {'name': '(a) Σs/Σt = 0.2', 'scatter_prob': 0.2},
    {'name': '(b) Σs/Σt = 0.8', 'scatter_prob': 0.8}
]

results_analog = []

for case in cases:
    print(f"\nCase: {case['name']}")
    print(f"{'θ (deg)':<8} {'N histories':<14} {'Transmission (%)':<18} {'Rtrans':<10} "
          f"{'Reflection (%)':<18} {'Rrefl':<10} {'Absorption (%)':<18} {'Rabs':<10} {'FOM':<10}")
    print("-"*120)

    for mu, theta in zip(mus, angles_deg):

        res = run_adaptive(
            thickness=thickness,
            sigma_t=sigma_t,
            scatter_prob=case['scatter_prob'],
            mu_source=mu,
            max_histories=10_000_000,
            target_R=0.10
        )

        results_analog.append({
            'Case': case['name'],
            'Theta': theta,
            'Mu': mu,
            'N_histories': res['N'],
            'Transmission': res['P_T']*100,
            'R_trans': res['R_T'],
            'Reflection': res['P_R']*100,
            'R_refl': res['R_R'],
            'Absorption': res['P_A']*100,
            'R_abs': res['R_A'],
            'FOM': res['FOM'],
            'Time': res['Time']
        })

        print(f"{theta:<8.1f} {res['N']:<14,} {res['P_T']*100:<18.4f} {res['R_T']:<10.4f} "
              f"{res['P_R']*100:<18.4f} {res['R_R']:<10.4f} {res['P_A']*100:<18.4f} {res['R_A']:<10.4f} "
              f"{res['FOM']:<10.2f}")

Discussion

- As θ increases and μ decreases, transmission decreases because particles travel a longer effective path through the shield ($d/\mu$). Example: At θ=60° (μ=0.5), the effective thickness is around 2× the physical thickness.

- Higher Σs/Σt (0.8 vs 0.2) results in higher transmission because scattering allows particles to random-walk toward the exit. But high scattering also increases the chance of back-scattering (reflection).

- Rare events such as transmission at θ=0°, Σs/Σt=0.2 (transmission at large angles and low scattering ratios) have high relative uncertainty even at large N. So, the stopping criterion (R ≤ 10%) required more histories for high-scattering cases.

- FOM is highest for cases with moderate probabilities (P ~ 0.1–0.5) because R is minimized. Example: at θ=30°, FOM = 598.70 for Σs/Σt=0.8 while FOM = 119.95 for Σs/Σt=0.2.

# Question 4

In [ ]:
@njit
def run_transport_implicit(n_particles, thickness, sigma_t, scatter_prob, mu_source):
    """
    Implicit capture (survival biasing):
    - Particles always scatter (never absorbed)
    - Weight reduced by scatter_prob at each collision
    - Tallies use weight instead of binary count
    """
    w_trans, w_ref, w_abs = 0.0, 0.0, 0.0
    max_steps = 10000

    for _ in range(n_particles):
        x = 0.0
        mu = mu_source
        weight = 1.0
        alive = True
        steps = 0

        while alive and steps < max_steps:
            steps += 1

            # Check exits
            if mu > 0 and x >= thickness:
                w_trans += weight; alive = False; continue
            elif mu < 0 and x <= 0:
                w_ref += weight; alive = False; continue

            # Sample distance
            xi = np.random.uniform(1e-10, 1.0)
            dist = -np.log(xi) / sigma_t
            x_new = x + dist * mu

            # Boundary checks
            if mu > 0 and x_new >= thickness:
                w_trans += weight; alive = False; continue
            elif mu < 0 and x_new <= 0:
                w_ref += weight; alive = False; continue

            # Collision: implicit capture
            x = x_new
            w_abs += weight * (1.0 - scatter_prob)
            weight *= scatter_prob

            # Isotropic
            mu = np.random.uniform(-1, 1)

            # Russian roulette for very low weights
            if weight < 1e-6:
                if np.random.uniform() < weight * 1e6:
                    weight = 1e-6  # Survive with boosted weight
                else:
                    alive = False  # Terminate

    return w_trans, w_ref, w_abs

In [ ]:
def run_adaptive_implicit(thickness, sigma_t, scatter_prob, mu_source,
                          max_histories=10_000_000, target_R=0.10, batch=50_000):

    w_trans, w_ref, w_abs = 0.0, 0.0, 0.0
    n_done = 0
    start_time = time.time()

    while n_done < max_histories:
        wt, wr, wa = run_transport_implicit(batch, thickness, sigma_t, scatter_prob, mu_source)
        w_trans += wt
        w_ref += wr
        w_abs += wa
        n_done += batch

        # Convert weights to probabilities
        total_weight = w_trans + w_ref + w_abs
        if total_weight == 0:
            continue

        p_t = w_trans / total_weight
        p_r = w_ref / total_weight
        p_a = w_abs / total_weight

        r_t = calc_relative_uncertainty(p_t, n_done)
        r_r = calc_relative_uncertainty(p_r, n_done)
        r_a = calc_relative_uncertainty(p_a, n_done)
        r_max = max(r_t, r_r, r_a)

        if r_max <= target_R:
            break

    elapsed = time.time() - start_time
    total_weight = w_trans + w_ref + w_abs
    p_t = w_trans / total_weight if total_weight > 0 else 0
    p_r = w_ref / total_weight if total_weight > 0 else 0
    p_a = w_abs / total_weight if total_weight > 0 else 0

    r_t = calc_relative_uncertainty(p_t, n_done)
    r_r = calc_relative_uncertainty(p_r, n_done)
    r_a = calc_relative_uncertainty(p_a, n_done)
    r_max = max(r_t, r_r, r_a)
    fom = calc_fom(r_max, elapsed)

    return {
        'N': n_done, 'P_T': p_t, 'P_R': p_r, 'P_A': p_a,
        'R_T': r_t, 'R_R': r_r, 'R_A': r_a, 'R_max': r_max,
        'FOM': fom, 'Time': elapsed
    }

In [ ]:
results_implicit = []

for case in cases:
    print(f"\nCase: {case['name']} (Implicit Capture)")

    print(f"{'θ (deg)':<8} {'N histories':<14} {'Transmission (%)':<18} {'Rtrans':<10} "
          f"{'Reflection (%)':<18} {'Rrefl':<10} {'Absorption (%)':<18} {'Rabs':<10} {'FOM':<10}")
    print("-"*120)


    for mu, theta in zip(mus, angles_deg):
        res = run_adaptive_implicit(
            thickness=thickness,
            sigma_t=sigma_t,
            scatter_prob=case['scatter_prob'],
            mu_source=mu,
            max_histories=10_000_000,
            target_R=0.10
        )

        results_implicit.append({
            'Case': case['name'],
            'Theta': theta,
            'Mu': mu,
            'N_histories': res['N'],
            'Transmission': res['P_T']*100,
            'R_trans': res['R_T'],
            'Reflection': res['P_R']*100,
            'R_refl': res['R_R'],
            'Absorption': res['P_A']*100,
            'R_abs': res['R_A'],
            'FOM': res['FOM'],
            'Time': res['Time']
        })

        print(f"{theta:<8.1f} {res['N']:<14,} {res['P_T']*100:<18.4f} {res['R_T']:<10.4f} "
              f"{res['P_R']*100:<18.4f} {res['R_R']:<10.4f} {res['P_A']*100:<18.4f} {res['R_A']:<10.4f} "
              f"{res['FOM']:<10.2f}")

Discussion:
- Implicit capture reduced the number of histories needed to reach R≤10% for transmission estimates, especially in deep-penetration cases (θ=0°, high Σt·d).

- FOM increased compared to analog simulation, it shows that implicit capture is more computationally efficient for this problem.

- Transmission/reflection/absorption estimates from implicit capture agree with analog results within statistical uncertainty, it confirms that the method is unbiased.

- For reflection estimates (often dominated by early collisions), implicit capture provides less benefit because absorption at the front surface is still a "rare event" in weighted terms.

- Implicit capture is effective for problems with high optical thickness (Σt·d = 10). But for problems with significant absorption near the source, analog Monte Carlo is reasonable enough to provide results.